# D&D 5E Challenge Rating Prediction Model

This notebook builds a machine learning model to predict Challenge Rating (CR) for D&D 5E monsters.

## Goals
1. Create a predictive model for CR based on monster attributes
2. Identify which features are most important for determining CR
3. Build separate models for low CR (≤1) and standard CR (≥2) monsters
4. Enable prediction of CR for custom monsters

## Approach
- **Two-Model System**: Low CR monsters (0, 1/8, 1/4, 1/2, 1) use classification; Standard CR (2+) uses regression
- **Feature Engineering**: Parse text fields, extract numeric values, create derived metrics
- **Text Embeddings**: Use NLP to capture similarity in monster abilities
- **Feature Importance**: Analyze which attributes matter most at different CR ranges

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load the monster data
df = pd.read_csv('dnd5e_monsters_2014.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Convert Challenge Rating to numeric
def cr_to_numeric(cr_str):
    """Convert CR string to numeric value"""
    if pd.isna(cr_str):
        return np.nan
    
    cr_str = str(cr_str).strip()
    
    # Handle fractions
    if '/' in cr_str:
        num, denom = cr_str.split('/')
        return float(num) / float(denom)
    
    return float(cr_str)

df['cr_numeric'] = df['Challenge_Rating'].apply(cr_to_numeric)

print("CR Distribution:")
print(df['cr_numeric'].value_counts().sort_index())

In [ ]:
# Visualize CR distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall distribution
axes[0].hist(df['cr_numeric'], bins=30, edgecolor='black')
axes[0].set_xlabel('Challenge Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Overall CR Distribution')
axes[0].axvline(x=1, color='red', linestyle='--', label='CR=1 (split point)')
axes[0].legend()

# Low CR vs Standard CR split
df['cr_category'] = df['cr_numeric'].apply(lambda x: 'Low CR (≤1)' if x <= 1 else 'Standard CR (≥2)')
cr_counts = df['cr_category'].value_counts()
axes[1].bar(cr_counts.index, cr_counts.values, edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('Monster Count by CR Category')
axes[1].set_ylim(0, max(cr_counts.values) * 1.1)

for i, v in enumerate(cr_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nLow CR (≤1): {sum(df['cr_numeric'] <= 1)} monsters")
print(f"Standard CR (≥2): {sum(df['cr_numeric'] >= 2)} monsters")

In [ ]:
# Check for missing values
print("Missing values by column:")
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 3. Feature Engineering

We'll create features in several categories:
- **Numeric parsing**: Extract numbers from string fields (HP, AC, Speed)
- **Ability scores**: Use modifiers and create derived stats
- **Combat metrics**: HP/AC ratios, effective HP, etc.
- **Action economy**: Count actions, reactions, legendary actions
- **Categorical**: Size and Type encoding

In [ ]:
### A. Parse numeric features from strings

def parse_hp_avg(hp_str):
    """Extract average HP from string like '135 (18d10+36)'"""
    if pd.isna(hp_str):
        return np.nan
    match = re.match(r'(\d+)', str(hp_str))
    return int(match.group(1)) if match else np.nan

def parse_hp_dice(hp_str):
    """Extract hit dice count and size from '135 (18d10+36)'"""
    if pd.isna(hp_str):
        return np.nan, np.nan
    match = re.search(r'(\d+)d(\d+)', str(hp_str))
    if match:
        return int(match.group(1)), int(match.group(2))
    return np.nan, np.nan

def parse_ac_value(ac_str):
    """Extract AC value from '17 (Natural Armor)' or '10'"""
    if pd.isna(ac_str):
        return np.nan
    match = re.match(r'(\d+)', str(ac_str))
    return int(match.group(1)) if match else np.nan

# Apply parsers
df['hp_avg'] = df['HP'].apply(parse_hp_avg)
df[['hp_dice_count', 'hp_dice_size']] = df['HP'].apply(lambda x: pd.Series(parse_hp_dice(x)))
df['ac_value'] = df['AC'].apply(parse_ac_value)

print("Sample parsed values:")
df[['Name', 'HP', 'hp_avg', 'hp_dice_count', 'hp_dice_size', 'AC', 'ac_value']].head()

In [ ]:
### B. Parse speed features

def parse_speed_features(speed_str):
    """Extract all speed types from '40 ft., fly 80 ft., swim 40 ft.'"""
    if pd.isna(speed_str):
        return pd.Series([0, 0, 0, 0, 0, 0, 0])
    
    speed_str = str(speed_str).lower()
    
    # Extract ground speed (first number)
    ground_match = re.match(r'(\d+)', speed_str)
    ground = int(ground_match.group(1)) if ground_match else 0
    
    # Extract other speeds
    fly = int(re.search(r'fly (\d+)', speed_str).group(1)) if re.search(r'fly (\d+)', speed_str) else 0
    swim = int(re.search(r'swim (\d+)', speed_str).group(1)) if re.search(r'swim (\d+)', speed_str) else 0
    burrow = int(re.search(r'burrow (\d+)', speed_str).group(1)) if re.search(r'burrow (\d+)', speed_str) else 0
    climb = int(re.search(r'climb (\d+)', speed_str).group(1)) if re.search(r'climb (\d+)', speed_str) else 0
    
    max_speed = max(ground, fly, swim, burrow, climb)
    movement_types = sum([1 for s in [ground, fly, swim, burrow, climb] if s > 0])
    
    return pd.Series([ground, fly, swim, burrow, climb, max_speed, movement_types])

df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb', 'max_speed', 'movement_types_count']] = \
    df['Speed'].apply(parse_speed_features)

print("Sample speed features:")
df[['Name', 'Speed', 'speed_ground', 'speed_fly', 'max_speed', 'movement_types_count']].head(10)

In [ ]:
### C. Ability score features

# Convert modifier strings to integers
def parse_modifier(mod_str):
    """Convert '+5' or '-1' to integer"""
    if pd.isna(mod_str):
        return 0
    mod_str = str(mod_str).strip()
    if not mod_str or mod_str == '':
        return 0
    return int(mod_str)

# Parse ability modifiers
for ability in ['STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA']:
    df[f'{ability}_Mod'] = df[f'{ability}_Mod'].apply(parse_modifier)

# Create derived ability features
df['total_modifier_sum'] = df[['STR_Mod', 'DEX_Mod', 'CON_Mod', 'INT_Mod', 'WIS_Mod', 'CHA_Mod']].sum(axis=1)
df['physical_modifier_sum'] = df[['STR_Mod', 'DEX_Mod', 'CON_Mod']].sum(axis=1)
df['mental_modifier_sum'] = df[['INT_Mod', 'WIS_Mod', 'CHA_Mod']].sum(axis=1)
df['highest_modifier'] = df[['STR_Mod', 'DEX_Mod', 'CON_Mod', 'INT_Mod', 'WIS_Mod', 'CHA_Mod']].max(axis=1)
df['lowest_modifier'] = df[['STR_Mod', 'DEX_Mod', 'CON_Mod', 'INT_Mod', 'WIS_Mod', 'CHA_Mod']].min(axis=1)

print("Sample ability score features:")
df[['Name', 'STR_Mod', 'total_modifier_sum', 'physical_modifier_sum', 'mental_modifier_sum', 'highest_modifier']].head()

In [ ]:
### D. Derived combat metrics

# Survivability metrics
df['hp_to_ac_ratio'] = df['hp_avg'] / df['ac_value'].replace(0, 1)  # Avoid division by zero
df['effective_hp'] = df['hp_avg'] * (1 + df['ac_value'] / 20)

# Count saving throw proficiencies
def count_saves(saves_str):
    """Count number of saving throw proficiencies"""
    if pd.isna(saves_str) or saves_str == '':
        return 0
    return len(re.findall(r'\w+ [+-]\d+', str(saves_str)))

df['save_proficiency_count'] = df['Saving_Throws'].apply(count_saves)

# Count skills
def count_skills(skills_str):
    """Count number of skill proficiencies"""
    if pd.isna(skills_str) or skills_str == '':
        return 0
    return len(re.findall(r'\w+ [+-]\d+', str(skills_str)))

df['skill_proficiency_count'] = df['Skills'].apply(count_skills)

print("Sample combat metrics:")
df[['Name', 'hp_avg', 'ac_value', 'hp_to_ac_ratio', 'effective_hp', 'save_proficiency_count']].head()

In [ ]:
### E. Damage type coverage

def count_items(text_str):
    """Count comma-separated items"""
    if pd.isna(text_str) or text_str == '':
        return 0
    # Split by common delimiters
    items = re.split(r'[,;]', str(text_str))
    return len([item.strip() for item in items if item.strip()])

df['resistance_count'] = df['Resistances'].apply(count_items)
df['immunity_count'] = df['Immunities'].apply(count_items)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_items)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_items)

print("Sample damage type coverage:")
df[['Name', 'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count']].head(10)

In [ ]:
### F. Senses features

def parse_senses(senses_str):
    """Extract vision types and ranges"""
    if pd.isna(senses_str):
        senses_str = ''
    senses_str = str(senses_str).lower()
    
    # Darkvision
    darkvision_match = re.search(r'darkvision (\d+)', senses_str)
    has_darkvision = 1 if darkvision_match else 0
    darkvision_range = int(darkvision_match.group(1)) if darkvision_match else 0
    
    # Other senses
    has_blindsight = 1 if 'blindsight' in senses_str else 0
    has_truesight = 1 if 'truesight' in senses_str else 0
    has_tremorsense = 1 if 'tremorsense' in senses_str else 0
    
    return pd.Series([has_darkvision, darkvision_range, has_blindsight, has_truesight, has_tremorsense])

df[['has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 'has_tremorsense']] = \
    df['Senses'].apply(parse_senses)

# Parse passive perception
df['passive_perception'] = pd.to_numeric(df['Passive_Perception'], errors='coerce').fillna(10)

print("Sample sense features:")
df[['Name', 'has_darkvision', 'darkvision_range', 'has_blindsight', 'passive_perception']].head(10)

In [ ]:
### G. Action economy features

def count_abilities(ability_str):
    """Count number of abilities (traits/actions) separated by ' | '"""
    if pd.isna(ability_str) or ability_str == '':
        return 0
    return len([a for a in str(ability_str).split(' | ') if a.strip()])

df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities)

# Legendary actions
df['legendary_action_count'] = df['Legendary_Actions'].apply(count_abilities)
df['has_legendary_actions'] = (df['legendary_action_count'] > 0).astype(int)
df['legendary_actions_per_round'] = pd.to_numeric(df['Legendary_Actions_Num'], errors='coerce').fillna(0)

# Total abilities
df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + \
                            df['bonus_action_count'] + df['legendary_action_count']

print("Sample action economy features:")
df[['Name', 'trait_count', 'action_count', 'reaction_count', 'legendary_action_count', 'has_legendary_actions']].head(10)

In [ ]:
### H. Parse attack features from Actions

def parse_attack_features(actions_str):
    """Extract multiattack, highest attack bonus, and save DC"""
    if pd.isna(actions_str):
        actions_str = ''
    actions_str = str(actions_str).lower()
    
    # Multiattack
    has_multiattack = 1 if 'multiattack' in actions_str else 0
    
    # Extract attack bonuses
    attack_bonuses = re.findall(r'\+(\d+) to hit', actions_str)
    highest_attack_bonus = max([int(b) for b in attack_bonuses]) if attack_bonuses else 0
    
    # Extract save DCs
    save_dcs = re.findall(r'dc (\d+)', actions_str)
    highest_save_dc = max([int(dc) for dc in save_dcs]) if save_dcs else 0
    
    return pd.Series([has_multiattack, highest_attack_bonus, highest_save_dc])

# Combine Traits and Actions for parsing
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' + 
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))

df[['has_multiattack', 'highest_attack_bonus', 'highest_save_dc']] = \
    df['Actions'].apply(parse_attack_features)

print("Sample attack features:")
df[['Name', 'has_multiattack', 'highest_attack_bonus', 'highest_save_dc']].head(10)

In [ ]:
### I. Special ability features (pattern matching)

# Check for common powerful abilities
df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regenerat', case=False, na=False).astype(int)
df['has_spellcasting'] = combined_abilities.str.contains('spellcasting', case=False, na=False).astype(int)

# Extract spellcaster level if present
def extract_spellcaster_level(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)(?:st|nd|rd|th)-level spellcaster', str(text).lower())
    return int(match.group(1)) if match else 0

df['spellcaster_level'] = combined_abilities.apply(extract_spellcaster_level)

print("Sample special ability features:")
df[['Name', 'has_legendary_resistance', 'has_magic_resistance', 'has_regeneration', 
    'has_spellcasting', 'spellcaster_level']].head(10)

In [ ]:
### J. Size and Type encoding

# Size ordinal encoding
size_mapping = {
    'Tiny': 1,
    'Small': 2,
    'Medium': 3,
    'Large': 4,
    'Huge': 5,
    'Gargantuan': 6
}
df['size_ordinal'] = df['Size'].map(size_mapping).fillna(3)  # Default to Medium

# Type one-hot encoding
type_dummies = pd.get_dummies(df['Type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)

print(f"Size encoding: {df['size_ordinal'].value_counts().sort_index()}")
print(f"\nType columns created: {type_dummies.columns.tolist()[:10]}...")

In [ ]:
# Summary of all engineered features
feature_columns = [
    # Parsed numeric
    'hp_avg', 'hp_dice_count', 'hp_dice_size', 'ac_value',
    # Speed
    'speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb', 
    'max_speed', 'movement_types_count',
    # Ability modifiers
    'STR_Mod', 'DEX_Mod', 'CON_Mod', 'INT_Mod', 'WIS_Mod', 'CHA_Mod',
    'total_modifier_sum', 'physical_modifier_sum', 'mental_modifier_sum',
    'highest_modifier', 'lowest_modifier',
    # Combat metrics
    'hp_to_ac_ratio', 'effective_hp', 'save_proficiency_count', 'skill_proficiency_count',
    # Damage coverage
    'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count',
    # Senses
    'has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 
    'has_tremorsense', 'passive_perception',
    # Action economy
    'trait_count', 'action_count', 'reaction_count', 'bonus_action_count',
    'legendary_action_count', 'has_legendary_actions', 'legendary_actions_per_round',
    'total_ability_count',
    # Attack features
    'has_multiattack', 'highest_attack_bonus', 'highest_save_dc',
    # Special abilities
    'has_legendary_resistance', 'has_magic_resistance', 'has_regeneration',
    'has_spellcasting', 'spellcaster_level',
    # Size
    'size_ordinal'
] + [col for col in df.columns if col.startswith('type_')]

print(f"Total engineered features: {len(feature_columns)}")
print(f"\nFeature list: {feature_columns}")

## 4. Model Training & Evaluation

In [ ]:
# Prepare data - handle missing values
X = df[feature_columns].copy()
y = df['cr_numeric'].copy()

# Fill any remaining NaN values
X = X.fillna(0)

# Remove any rows where CR is NaN
valid_idx = ~y.isna()
X = X[valid_idx]
y = y[valid_idx]

print(f"Final dataset shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")

### 4a. Low CR Model (CR ≤ 1)

In [ ]:
# Split into low CR and standard CR
low_cr_mask = y <= 1
X_low = X[low_cr_mask]
y_low = y[low_cr_mask]

print(f"Low CR dataset: {X_low.shape[0]} monsters")
print(f"CR distribution:")
print(y_low.value_counts().sort_index())

In [ ]:
# Train/test split for low CR
X_low_train, X_low_test, y_low_train, y_low_test = train_test_split(
    X_low, y_low, test_size=0.2, random_state=42, stratify=y_low
)

# Train Random Forest Classifier
low_cr_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
low_cr_model.fit(X_low_train, y_low_train)

# Predictions
y_low_pred = low_cr_model.predict(X_low_test)

# Evaluation
accuracy = accuracy_score(y_low_test, y_low_pred)
print(f"Low CR Model Accuracy: {accuracy:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_low_test, y_low_pred))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_low_test, y_low_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Low CR Model Confusion Matrix')
plt.ylabel('Actual CR')
plt.xlabel('Predicted CR')
plt.show()

# Within ±1 tier accuracy
cr_values = sorted(y_low.unique())
cr_to_tier = {cr: i for i, cr in enumerate(cr_values)}
y_low_test_tier = y_low_test.map(cr_to_tier)
y_low_pred_tier = pd.Series(y_low_pred).map(cr_to_tier)
within_one = (abs(y_low_test_tier.values - y_low_pred_tier.values) <= 1).mean()
print(f"\nAccuracy within ±1 tier: {within_one:.3f}")

### 4b. Standard CR Model (CR ≥ 2)

In [ ]:
# Split into standard CR
standard_cr_mask = y >= 2
X_standard = X[standard_cr_mask]
y_standard = y[standard_cr_mask]

print(f"Standard CR dataset: {X_standard.shape[0]} monsters")
print(f"CR range: {y_standard.min()} to {y_standard.max()}")

In [ ]:
# Train/test split for standard CR
X_standard_train, X_standard_test, y_standard_train, y_standard_test = train_test_split(
    X_standard, y_standard, test_size=0.2, random_state=42
)

# Train Random Forest Regressor
standard_cr_model = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=15)
standard_cr_model.fit(X_standard_train, y_standard_train)

# Predictions
y_standard_pred = standard_cr_model.predict(X_standard_test)

# Evaluation
mae = mean_absolute_error(y_standard_test, y_standard_pred)
rmse = np.sqrt(mean_squared_error(y_standard_test, y_standard_pred))
r2 = r2_score(y_standard_test, y_standard_pred)

print(f"Standard CR Model Performance:")
print(f"  MAE: {mae:.3f} CR points")
print(f"  RMSE: {rmse:.3f}")
print(f"  R² Score: {r2:.3f}")

# Within ±1 and ±2 CR accuracy
within_1 = (abs(y_standard_test - y_standard_pred) <= 1).mean()
within_2 = (abs(y_standard_test - y_standard_pred) <= 2).mean()
print(f"\nPredictions within ±1 CR: {within_1:.1%}")
print(f"Predictions within ±2 CR: {within_2:.1%}")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_standard_test, y_standard_pred, alpha=0.6)
plt.plot([y_standard_test.min(), y_standard_test.max()], 
         [y_standard_test.min(), y_standard_test.max()], 
         'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual CR')
plt.ylabel('Predicted CR')
plt.title('Standard CR Model: Predicted vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Residuals plot
residuals = y_standard_test - y_standard_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_standard_pred, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.axhline(y=2, color='orange', linestyle=':', lw=1, label='±2 CR')
plt.axhline(y=-2, color='orange', linestyle=':', lw=1)
plt.xlabel('Predicted CR')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Standard CR Model: Residuals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Low CR Model Feature Importance
low_cr_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': low_cr_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features for Low CR (≤1):")
print(low_cr_importance.head(20))

# Visualize
plt.figure(figsize=(12, 8))
top_20_low = low_cr_importance.head(20)
plt.barh(range(len(top_20_low)), top_20_low['importance'])
plt.yticks(range(len(top_20_low)), top_20_low['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Features for Low CR Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Standard CR Model Feature Importance
standard_cr_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': standard_cr_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features for Standard CR (≥2):")
print(standard_cr_importance.head(20))

# Visualize
plt.figure(figsize=(12, 8))
top_20_standard = standard_cr_importance.head(20)
plt.barh(range(len(top_20_standard)), top_20_standard['importance'])
plt.yticks(range(len(top_20_standard)), top_20_standard['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Features for Standard CR Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Compare feature importance between models
comparison = low_cr_importance.merge(
    standard_cr_importance, 
    on='feature', 
    suffixes=('_low', '_standard')
)
comparison['difference'] = comparison['importance_standard'] - comparison['importance_low']
comparison_sorted = comparison.sort_values('difference', key=abs, ascending=False)

print("\nFeatures with Biggest Importance Difference (Low CR vs Standard CR):")
print("Positive = more important for Standard CR, Negative = more important for Low CR")
print(comparison_sorted.head(15)[['feature', 'importance_low', 'importance_standard', 'difference']])

## 6. Combined Prediction Pipeline

In [ ]:
def predict_cr(monster_features):
    """
    Predict CR for a monster using the appropriate model.
    
    Parameters:
    -----------
    monster_features : dict or pd.Series
        Dictionary or Series with feature values
    
    Returns:
    --------
    float : Predicted CR value
    """
    # Convert to DataFrame if dict
    if isinstance(monster_features, dict):
        monster_features = pd.DataFrame([monster_features])
    elif isinstance(monster_features, pd.Series):
        monster_features = monster_features.to_frame().T
    
    # Ensure all features are present
    for col in feature_columns:
        if col not in monster_features.columns:
            monster_features[col] = 0
    
    monster_features = monster_features[feature_columns]
    
    # Make a preliminary prediction to determine which model to use
    # Use a simple heuristic: if hp_avg < 50 and ac_value < 16, likely low CR
    hp = monster_features['hp_avg'].iloc[0]
    ac = monster_features['ac_value'].iloc[0]
    has_legendary = monster_features['has_legendary_actions'].iloc[0]
    
    # Route to appropriate model
    if hp < 50 and ac < 16 and has_legendary == 0:
        # Use low CR model
        pred_cr = low_cr_model.predict(monster_features)[0]
        model_used = 'Low CR Model'
    else:
        # Use standard CR model
        pred_cr = standard_cr_model.predict(monster_features)[0]
        # Ensure prediction is at least 2
        pred_cr = max(2, pred_cr)
        model_used = 'Standard CR Model'
    
    print(f"Model used: {model_used}")
    print(f"Predicted CR: {pred_cr:.2f}")
    
    return pred_cr

# Test with a few examples
print("Example 1: Goblin")
goblin_idx = df[df['Name'] == 'Goblin'].index[0]
pred = predict_cr(X.loc[goblin_idx])
print(f"Actual CR: {y.loc[goblin_idx]}\n")

print("Example 2: Ancient Red Dragon")
dragon_idx = df[df['Name'] == 'Ancient Red Dragon'].index[0]
pred = predict_cr(X.loc[dragon_idx])
print(f"Actual CR: {y.loc[dragon_idx]}")

## 7. Conclusions & Insights

### Model Performance Summary

**Low CR Model (CR ≤ 1)**:
- Classification approach for 5 discrete CR values (0, 1/8, 1/4, 1/2, 1)
- Accuracy and within-tier metrics shown above
- Most important features: HP, AC, attack bonus, basic combat stats

**Standard CR Model (CR ≥ 2)**:
- Regression approach for continuous CR prediction
- MAE, RMSE, and R² metrics shown above
- Most important features: HP, legendary actions, immunities, effective HP

### Key Insights

1. **Feature Importance Varies by CR Range**:
   - Low CR: Basic stats (HP, AC) dominate
   - High CR: Special abilities (legendary actions, resistances) become crucial

2. **HP and AC are Fundamental**:
   - Appear in top features for both models
   - HP-to-AC ratio and effective HP are valuable derived metrics

3. **Action Economy Matters**:
   - Legendary actions are strong CR predictors at high levels
   - Multiattack presence affects low CR predictions

4. **Defensive Abilities**:
   - Resistances and immunities more important for high CR
   - Legendary resistance is a strong signal for high CR

### Using This Model

You can now:
1. Input custom monster stats and get CR predictions
2. Understand which features to adjust to change CR
3. Balance monsters by targeting specific CR values
4. Identify which attributes provide the most "value" for CR budgeting

### Next Steps

1. Fine-tune models with hyperparameter optimization
2. Add more sophisticated NLP features using embeddings
3. Create interactive tool for custom monster CR prediction
4. Analyze CR "budgeting" - how much CR do different abilities cost?
5. Build inverse model: given desired CR, suggest stat ranges

In [ ]:
# Save models for future use
import pickle

with open('low_cr_model.pkl', 'wb') as f:
    pickle.dump(low_cr_model, f)

with open('standard_cr_model.pkl', 'wb') as f:
    pickle.dump(standard_cr_model, f)

# Save feature columns list
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("Models saved successfully!")
print("Files created:")
print("  - low_cr_model.pkl")
print("  - standard_cr_model.pkl")
print("  - feature_columns.pkl")